In [2]:
# Add autoreload at the top of the notebook you're working on 
# in order for it to auto refresh when you change the 'project_package'
%load_ext autoreload
%autoreload 2

# Table of contents

[1. Import and align datasets for bias analysis](#section-1)   
&emsp; [a. Load the processed recipe dataset](#section-1a)  
&emsp; [b. Load user rating data](#section-1b)  
&emsp; [c. Create user preference data](#section-1c)<br>
&emsp; [d. Get the recipe vectorstores](#section-1d)<br>
&emsp; [e. Get collection for user preferences](#section-1e)

[2. Load trained recommendation models and Rectools dataset](#section-2) 

[3. Analyzing label distribution through recommendation pipeline](#section-3)   
&emsp; [a. Default label distribution](#section-3a)  
&emsp; [b. Label distribution with attempt to remove bias](#section-3b)<br>
&emsp; [c. Label distribution with attempt to add bias](#section-3c)

[4. Recommendation pipeline evaluation with sankey plot](#section-4)    
&emsp; [4.1. BM25 model](#section-4.1)<br>
&emsp;&emsp; [a. Default setting (no bias modification)](#section-4.1a)<br>
&emsp;&emsp; [b. Pipeline with attempt to remove bias](#section-4.1b)<br>
&emsp;&emsp; [c. Pipeline with attempt to add bias](#section-4.1c)<br>
&emsp; [4.2. SVD model](#section-4.2)<br>
&emsp; [4.3. ALS model](#section-4.3)<br> 
&emsp; [4.4. LightFM hybrid model](#section-4.4)



Import support libraries & modules

In [ ]:
import os
from pathlib import Path
import logging
import ast
logging.getLogger("httpx").setLevel(logging.WARNING)  # hide logging in cell when using chroma vectorstore
from dotenv import load_dotenv

import plotly.express as px
import pandas as pd
from flashrank import Ranker
from rectools.models import load_model

from project_package.data_preprocessing.utils import create_user_preference
from project_package.modeling.recommendation_utils import (
    construct_rec_train_dataset,get_embedding_model,
    VectorstoreLoader,doc_template_fill_in,keep_n_labels,
    recommendation_doc_id_pipeline,load_vector_store
    )
from project_package.visualization import sankey_plot
from project_package.data_preprocessing.default import (
    USER,ITEM,RECIPE_TO_USE,
    RECIPE_COLUMN_MAPPING,REVIEW_COLUMN_MAPPING,REVIEW_TO_USE,
    DOC_TEMPLATE,USER_PROFILE_TEMPLATE,RECIPE_COLS,RECIPE_META_COLS,PREFERENCE_COLS
    )
from project_package.aws.data_access import pandas_sql_df
from project_package.aws.model_store import get_object_bytes

load_dotenv()  # load env variables from .evn
root_directory = Path(os.getcwd()).parent  #NOTE: update of notebook location changed

g:\Python\envs\capstone_test3\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


<a id='section-1'></a>
## 1. Import and align datasets for bias analysis


a. Load the processed recipe dataset
<a id='section-1a'></a>

In [6]:
#NOTE: might need update to retrieve the processed df from S3

recipe_df = pd.read_csv(root_directory / "data/processed/Recipes/recipes_chunk_0.csv",usecols=RECIPE_TO_USE)
# recipe_df = load_chunks(root_directory / "data/processed/Recipes","recipes_chunk_*.csv",usecols=to_use_columns)  # uncomment for full dataset
recipe_df.rename(columns = RECIPE_COLUMN_MAPPING,inplace=True)

recipe_df['ingredients'] = recipe_df['ingredients'].apply(ast.literal_eval)  # convert ingredient to list

recipe_category_df = pandas_sql_df("SELECT * from RECIPES")
recipe_category_df['original_id'] = recipe_category_df['original_id'].astype(int)
recipe_category_df['cooking_method'] = recipe_category_df['cooking_method'].str.strip("{}").str.split(",")

recipe_df = recipe_df.merge(recipe_category_df,on=['original_id','source'],how='inner')
recipe_df = recipe_df.dropna(subset = RECIPE_META_COLS)
recipe_df = recipe_df[recipe_df["ingredients"].apply(lambda x: x != [])]  # remove empty ingredient record

#Drop recipe that take too long, or calories value that are unreasonably high for a common meal
recipe_df = recipe_df.loc[(recipe_df['total_time']<=1440)&(recipe_df['calories']<= 5000)].reset_index(drop=True) 

recipe_df.head(2)

,original_id,recipe_name,instructions,calories,source,prep_time,cook_time,total_time,ingredients,who_score,...,recipe_id,cuisine,cooking_method,difficulty,protein_content,fiber_content,fat_content,carbohydrate_content,sodium_content,s3_key
0,39,Biryani,['Soak saffron in warm milk for 5 minutes and ...,1110.7,foodcom,240,25.0,265,"[saffron, milk, green chili, onion, garlic, ga...",1,...,2,asian,"[braise, simmer, fry, bake]",advanced,high,high,high,high,medium,recipes/2.json
1,40,Best Lemonade,"['Into a 1 quart Jar with tight fitting lid, p...",311.1,foodcom,30,5.0,35,"[sugar, lemon zest, water, lemon juice]",3,...,1,unknown,[unknown],intermediate,low,low,low,high,low,recipes/1.json


b. Load user rating data
<a id='section-1b'></a>

In [7]:
# should sort reviews by user first

#NOTE: might need update to retrieve the processed df from S3, remove any users that have less than n reviews
user_reviews = pd.read_csv(root_directory / "data/processed/Reviews/reviews_chunk_0.csv",usecols=REVIEW_TO_USE)
# recipe_df = load_chunks(root_directory / "data/processed/Reviews","reviews_chunk_*.csv",usecols=to_use_columns)  # uncomment for full dataset
user_reviews.rename(columns = REVIEW_COLUMN_MAPPING,inplace=True)

# Convert datetime data
user_reviews['modified_time'] = pd.to_datetime(user_reviews['modified_time'], utc=True)
user_reviews['modified_time'] = user_reviews['modified_time'].dt.tz_localize(None)

# merge with recipe_df to retrieve the unique recipe_id from Postgres database
user_reviews = user_reviews.merge(recipe_df[['original_id','source','recipe_id']],on=['original_id','source'],how='inner')
user_reviews = user_reviews.sort_values(['source','original_user_id']).reset_index(drop=True)

# Convert to user from difference source to unique IDs
user_reviews["user_id"] = user_reviews["original_user_id"].astype(str) + "_" + user_reviews["source"]

groupby = user_reviews.groupby('user_id')['rating'].count()
drop_index = groupby.index[groupby<10]  #NOTE: threshold for to keep user with this minimum number of reviews

user_reviews = user_reviews.loc[~(user_reviews['user_id'].isin(drop_index))].reset_index(drop=True)
user_reviews["user_id"], _ = pd.factorize(user_reviews["user_id"])

user_reviews = user_reviews[['user_id','recipe_id','rating','modified_time']]
user_reviews.head(3)

,user_id,recipe_id,rating,modified_time
0,0,14063,4,2002-02-19 12:32:18
1,0,17605,5,2005-02-05 15:06:54
2,0,7593,5,2002-05-02 14:20:30


c. Retrieve user preference data
<a id='section-1c'></a>

In [8]:
preference_df = pd.read_csv(root_directory / 'data/processed/user_preferences.csv')
preference_df.head(3)

,user_id,ingredients,cuisine,cooking_method,difficulty,protein_content,fiber_content,fat_content,carbohydrate_content,sodium_content
0,0,"['butter', 'salt']",['american'],['bake'],['intermediate'],['low'],['low'],[],[],['medium']
1,1,"['butter', 'egg', 'salt', 'sugar']",['american'],['bake'],['intermediate'],['low'],['low'],[],['high'],['medium']
2,2,[],['american'],[],['intermediate'],[],['low'],[],[],[]


In [9]:
embedding_model = get_embedding_model(
    huggingface_model_path="BAAI/bge-small-en-v1.5",  # NOTE: Change this embedding to foodbert later if necessary
    local_model_name="bge-small",
    device="cuda"
)
chroma_path = root_directory / "data/processed/chroma_db"  #NOTE: Change the Path if necessary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

d. Get the recipe vectorstores
<a id='section-1d'></a>

In [10]:
#NOTE: Run the cell again when a batch is failed to continue

store_document = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_document:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="recipe_collection",  #NOTE: Change the collection name if necessary,
            embedding = embedding_model,
            doc_template = DOC_TEMPLATE,
            input_data = recipe_df,
            format_cols = RECIPE_COLS,
            meta_cols = RECIPE_META_COLS, # include item ID for later filter tasks
            persist_directory = chroma_path,
            docID_col = ITEM
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    vectorstore = load_vector_store(
        collection_name="recipe_collection",
        embedding_model=embedding_model,
        persist_directory=chroma_path
    )

Couldn't load the cloud Chroma vectorstore, switching to local storage.


e. Get collection for user preferences
<a id='section-1e'></a>

In [11]:
#NOTE: Run the cell again when a batch is failed to continue

store_user_pref = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_user_pref:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="user_recipe_preference",  #NOTE: Change the collection name if necessary,
            embedding = embedding_model,
            doc_template = USER_PROFILE_TEMPLATE,
            input_data = preference_df,
            format_cols = PREFERENCE_COLS,
            meta_cols = None,
            persist_directory = chroma_path,
            docID_col = USER
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        user_vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    user_vectorstore = load_vector_store(
        collection_name="user_recipe_preference",
        embedding_model=embedding_model,
        persist_directory=chroma_path
    )

Couldn't load the cloud Chroma vectorstore, switching to local storage.


## 2. Load trained recommendation models and Rectools dataset
<a id='section-2'></a>

In [12]:
k = 10  # number of recommendation to create

# load data to Rectools format
dataset = construct_rec_train_dataset(
    user_reviews,
    recipe_df,
    preference_df,
    use_datetime = True
)

In [ ]:
#NOTE: Load the models, for big model, we might need to load this from S3 or google drive
svd_model = load_model(root_directory / "models/recommendation_models/svd_recommendation_model.pkl")
als_model = load_model(root_directory / "models/recommendation_models/als_recommendation_model.pkl")
lightfm_model = load_model(root_directory / "models/recommendation_models/lightFM_recommendation_model.pkl")

# can LOAD model from S3 instead if it's there
# s3_client = build_s3_client(
#     os.environ['AWS_ACCESS_KEY'],
#     os.environ['AWS_SECRET_KEY'],
#     os.environ['AWS_SESSION_TOKEN']
# )
# svd_model = load_model(io.BytesIO(get_object_bytes(s3_client,'svd_recommendation_model')))
# als_model = load_model(io.BytesIO(get_object_bytes(s3_client,'als_recommendation_model')))
# lightfm_model = load_model(io.BytesIO(get_object_bytes(s3_client,'lightfm_recommendation_model')))

<a id='section-3'></a>
## 3. Analyzing label distribution through recommendation pipeline

First we will create some random queries to check the recommendation pipeline output

In [25]:
# a random sample of items to be used as queries
sample_df = recipe_df.sample(20,random_state=0)
sample_documents = doc_template_fill_in(DOC_TEMPLATE,sample_df,RECIPE_COLS,docID_col=ITEM)
sample_documents = [doc.page_content for doc in sample_documents]

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)

Then we look at the label distribution of a category before any filtering step

In [12]:
fig = px.histogram(
    keep_n_labels(recipe_df,'cooking_method',True,10), 
    x='cooking_method',
    log_y=True,
    height=400,
    title='Top 10 cooking methods in the full dataset'
    )
fig.update_xaxes(categoryorder='total descending').show()

<a id='section-3a'></a>
a. Default label distribution

Now we look at the dsitrbution of the feature labels after each recommendation step. We are using multiple different queries to give a more statistical significant view.

In [13]:
plot_df = pd.DataFrame()

for query in sample_documents:
    buffer_df, _ = recommendation_doc_id_pipeline(
        recipe_df,
        vectorstore,
        reranker,
        query,
        embedding_model=embedding_model,
        features = ['cooking_method']
    )
    plot_df = pd.concat((plot_df,buffer_df))

plot_df = plot_df.reset_index(drop=True)

In [14]:
fig = px.histogram(
    keep_n_labels(plot_df,'cooking_method',True,10), 
    x="cooking_method",
    color='pipeline_step', barmode='group',log_y=True,
    height=600,
    width=1000,
    title='Top 10 label after recommendation pipeline')
fig.update_xaxes(categoryorder='total descending').show()

<a id='section-3b'></a>
b. Label distribution with attempt to remove bias

We try to remove certain biases with weight from the queries and see how it performs

In [15]:
no_bias_df = pd.DataFrame()

for query in sample_documents:
    buffer_df, _ = recommendation_doc_id_pipeline(
        recipe_df,
        vectorstore,
        reranker,
        query,
        remove_biases = [("bake,simmer",1.5)],
        embedding_model=embedding_model,
        features = ['cooking_method'],
        neg_rank_bias="bake,simmer"
    )
    no_bias_df = pd.concat((no_bias_df,buffer_df))

no_bias_df = no_bias_df.reset_index(drop=True)

In [16]:
fig = px.histogram(
    keep_n_labels(no_bias_df,'cooking_method',True,10), 
    x="cooking_method",
    color='pipeline_step', barmode='group',log_y=True,
    height=600,
    width=1000,
    title='Top 10 label after recommendation pipeline with removing bias')
fig.update_xaxes(categoryorder='total descending').show()

<a id='section-3c'></a>
c. Label distribution with attempt to add bias

Now, for another case, we try to add some bias to the query to shift the recommendations

In [17]:
add_bias_df = pd.DataFrame()

for query in sample_documents:
    buffer_df, _ = recommendation_doc_id_pipeline(
        recipe_df,
        vectorstore,
        reranker,
        query,
        add_biases = [("stir_fry and grill",1.5)],
        embedding_model=embedding_model,
        features = ['cooking_method'],
        pos_rank_bias = "stir_fry and grill"
    )
    add_bias_df = pd.concat((add_bias_df,buffer_df))

add_bias_df = add_bias_df.reset_index(drop=True)

In [18]:
fig = px.histogram(
    keep_n_labels(add_bias_df,'cooking_method',True,10), 
    x="cooking_method",
    color='pipeline_step', barmode='group',log_y=True,
    height=600,
    width=1000,
    title='Top 10 label after recommendation pipeline with added bias')
fig.update_xaxes(categoryorder='total descending').show()

<a id='section-4'></a>
# 4. Recommendation pipeline evaluation with sankey plot

We will use a custom query to check how a category labels is being updated through recommendation pipeline

In [13]:
# a single test query
test_query = "spaghetti with meat and vegetable, tomatoes"

In [14]:
test_user_profile = USER_PROFILE_TEMPLATE.format(*preference_df.iloc[3][PREFERENCE_COLS].tolist())
print(test_user_profile)


Favorite ingredients are: ['butter', 'egg', 'flour', 'salt', 'sugar'].
Favorite cuisine are: ['american'].
Preferred cooking method: ['bake', 'simmer'].
Preferred cooking difficulty: ['intermediate'].
Preferred Protein content is ['low'].
Preferred Fiber content is ['low'].
Preferred Fat content is ['high'].
Preferred Carbohydrate content is ['high'].
Preferred Sodium content is ['high'].



<a id='section-4.1'></a>
### 4.1. BM25 model

We are testing a single query for a BM25 recommendation pipeline with default setting, removing bias, and adding bias

<a id='section-4.1a'></a>
a. Default setting (no bias modification)

In [21]:
# Single query pipeline test
test_query_df,_ = recommendation_doc_id_pipeline(
    recipe_df,
    vectorstore,
    reranker,
    test_query,
    embedding_model=embedding_model,
    n_recommendations = 200,
    features = ['ingredients'],
    include_fulldata = True
)

test_query_df = keep_n_labels(test_query_df,'ingredients',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'ingredients',"Sankey plot for category labels through pipeline",True,['garlic','olive oil','egg','butter'])

<a id='section-4.1b'></a>
b. Pipeline with attempt to remove bias

Same as previous section, we will try to remove some bias from the semantic search

In [22]:
# Single query pipeline test
no_bias_df,_ = recommendation_doc_id_pipeline(
    recipe_df,
    vectorstore,
    reranker,
    test_query,
    embedding_model=embedding_model,
    n_recommendations = 200,
    remove_biases = [("garlic and olive oil",1.5)],
    features = ['ingredients'],
    neg_rank_bias = "garlic and olive oil",
    include_fulldata = True
)
no_bias_df = keep_n_labels(no_bias_df,'ingredients',True,10)

# Plot the sankey plot
sankey_plot(no_bias_df,'ingredients',"Sankey plot for category labels through pipeline",True,['garlic','olive oil','egg','butter'])

<a id='section-4.1c'></a>
c. Pipeline with attempt to add bias

Then again, we check the difference in the sankey plot when we add some bias to the query

In [23]:
# Single query pipeline test
add_bias_df,_ = recommendation_doc_id_pipeline(
    recipe_df,
    vectorstore,
    reranker,
    test_query,
    embedding_model=embedding_model,
    n_recommendations = 200,
    add_biases = [("egg and butter",1.5)],
    features = ['ingredients'],
    pos_rank_bias = "egg and butter",
    include_fulldata = True
)
add_bias_df = keep_n_labels(add_bias_df,'ingredients',True,10)

# Plot the sankey plot
sankey_plot(add_bias_df,'ingredients',"Sankey plot for category labels through pipeline",True,['garlic','olive oil','egg','butter'])

We can also test other recommendation model to see how it performs

<a id='section-4.2'></a>
### 4.2. SVD model

In [24]:
test_query_df,test_rank = recommendation_doc_id_pipeline(
    recipe_df,
    vectorstore,
    reranker,
    test_query,
    dataset=dataset,
    embedding_model=embedding_model,
    n_recommendations = 200,
    recommendation_model=svd_model,
    user_vectorstore=user_vectorstore,
    user_profile=test_user_profile,
    model_type = 'collab',
    features = ['ingredients'],
    include_fulldata = True,
    random_state = 0
)

test_query_df = keep_n_labels(test_query_df,'ingredients',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'ingredients',"Sankey plot for SVD model",True,['garlic','olive oil','egg','butter'])

<a id='section-4.3'></a>
### 4.3. ALS model

In [25]:
test_query_df,test_rank = recommendation_doc_id_pipeline(
    recipe_df,
    vectorstore,
    reranker,
    test_query,
    dataset=dataset,
    embedding_model=embedding_model,
    n_recommendations = 200,
    recommendation_model=als_model,
    user_vectorstore=user_vectorstore,
    user_profile=test_user_profile,
    model_type = 'collab',
    features = ['ingredients'],
    include_fulldata = True,
    random_state = 0
)

test_query_df = keep_n_labels(test_query_df,'ingredients',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'ingredients',"Sankey plot for ALS model",True,['garlic','olive oil','egg','butter'])

<a id='section-4.4'></a>
### 4.4. LightFM hybrid model

In [26]:
test_query_df,test_rank = recommendation_doc_id_pipeline(
    recipe_df,
    vectorstore,
    reranker,
    test_query,
    dataset=dataset,
    embedding_model=embedding_model,
    n_recommendations = 200,
    recommendation_model=lightfm_model,
    user_vectorstore=user_vectorstore,
    user_profile=test_user_profile,
    model_type = 'hybrid',
    features = ['ingredients'],
    include_fulldata = True,
    random_state = 0
)

test_query_df = keep_n_labels(test_query_df,'ingredients',True,10)

# Plot the sankey plot
sankey_plot(test_query_df,'ingredients',"Sankey plot for LightFM model",True,['garlic','olive oil','egg','butter'])